In [22]:
!pip install -q kaggle

In [23]:
!kaggle datasets download -d abdallahalidev/plantvillage-dataset

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
plantvillage-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [24]:
import zipfile


with zipfile.ZipFile('plantvillage-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('plantvillage')

print("Extraction completed successfully!")

Extraction completed successfully!


In [25]:
import os

base_dir = 'plantvillage/plantvillage dataset/color'

classes = os.listdir(base_dir)

# Filter only the classes related to tomatoes
tomato_classes = [c for c in classes if 'Tomato' in c]

print(f"Total Tomato classes found: {len(tomato_classes)}\n")
print("Here are the tomato categories:")
for idx, tomato_class in enumerate(tomato_classes, 1):
    # Count how many images are in each tomato category folder
    class_path = os.path.join(base_dir, tomato_class)
    num_images = len(os.listdir(class_path))
    print(f"{idx}. {tomato_class} -> {num_images} images")

Total Tomato classes found: 10

Here are the tomato categories:
1. Tomato___Target_Spot -> 1404 images
2. Tomato___Bacterial_spot -> 2127 images
3. Tomato___Septoria_leaf_spot -> 1771 images
4. Tomato___Early_blight -> 1000 images
5. Tomato___Late_blight -> 1909 images
6. Tomato___healthy -> 1591 images
7. Tomato___Tomato_Yellow_Leaf_Curl_Virus -> 5357 images
8. Tomato___Tomato_mosaic_virus -> 373 images
9. Tomato___Leaf_Mold -> 952 images
10. Tomato___Spider_mites Two-spotted_spider_mite -> 1676 images


In [26]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define image size and batch size
IMG_SIZE = 224
BATCH_SIZE = 32

# Path to our tomato dataset folder
base_dir = 'plantvillage/plantvillage dataset/color'

# Create Data Generator with validation split (80% training, 20% validation)
datagen = ImageDataGenerator(
    rescale=1.0/255.0,      # Normalize pixel values from [0, 255] to [0, 1]
    validation_split=0.2    # 20% for validation
)

# Training data generator
train_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    subset='training',
    class_mode='categorical',
    classes=tomato_classes
)

# Validation data generator
val_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    subset='validation',
    class_mode='categorical',
    classes=tomato_classes
)

Found 14532 images belonging to 10 classes.
Found 3628 images belonging to 10 classes.


In [27]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

# 1. Load the pre-trained MobileNetV2 model (without its top classification layer)
base_model = MobileNetV2(input_shape=(224, 224, 3),
                         include_top=False,
                         weights='imagenet')

# Freeze the base model so its pre-trained weights won't be changed during early training
base_model.trainable = False

# 2. Build our custom model on top of it
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),         # Converts the 2D features into a 1D vector
    Dense(128, activation='relu'),    # Hidden layer to learn tomato-specific patterns
    Dense(10, activation='softmax')   # Output layer with 10 classes (probabilities for each tomato disease/health)
])

# 3. Compile the model
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("model is buil")

model is buil


In [28]:
# Train the model for 5 epochs
EPOCHS = 5

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

print("model training done")

Epoch 1/5
455/455 ━━━━━━━━━━━━━━━━━━━━ 711s 2s/step - accuracy: 0.8373 - loss: 0.4898 - val_accuracy: 0.8900 - val_loss: 0.3185
Epoch 2/5
455/455 ━━━━━━━━━━━━━━━━━━━━ 720s 2s/step - accuracy: 0.9139 - loss: 0.2549 - val_accuracy: 0.9055 - val_loss: 0.2765
Epoch 3/5
455/455 ━━━━━━━━━━━━━━━━━━━━ 723s 2s/step - accuracy: 0.9341 - loss: 0.1924 - val_accuracy: 0.9140 - val_loss: 0.2512
Epoch 4/5
455/455 ━━━━━━━━━━━━━━━━━━━━ 712s 2s/step - accuracy: 0.9498 - loss: 0.1496 - val_accuracy: 0.9107 - val_loss: 0.2710
Epoch 5/5
455/455 ━━━━━━━━━━━━━━━━━━━━ 730s 2s/step - accuracy: 0.9588 - loss: 0.1219 - val_accuracy: 0.9212 - val_loss: 0.2386
model training done


In [29]:
# Save the trained model using the modern native Keras format
model.save('tomato_disease_model.keras')

print("Saved")

Saved


In [30]:
!pip install -q streamlit streamlit-drawable-canvas

In [31]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import numpy as np
from PIL import Image

# Load our trained model
@st.cache_resource
def load_model():
    model = tf.keras.models.load_model('tomato_disease_model.keras')
    return model

model = load_model()

# List of the 10 tomato classes (in exact order used during training)
class_names = [
    'Tomato___Target_Spot',
    'Tomato___Bacterial_spot',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___healthy',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus',
    'Tomato___Leaf_Mold',
    'Tomato___Spider_mites Two-spotted_spider_mite'
]

# UI Design
st.title("🌿 Tomato Plant Disease Detection")
st.write("Upload an image of a tomato leaf to check if it's healthy or affected by a disease.")

uploaded_file = st.file_uploader("Choose a tomato leaf image...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # Display the uploaded image
    image_display = Image.open(uploaded_file)
    st.image(image_display, caption='Uploaded Tomato Leaf', use_container_width=True)

    st.write("Classifying...")

    # Preprocess the image for the model
    img = image_display.resize((224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) # Convert to batch dimension (1, 224, 224, 3)
    img_array = img_array / 255.0 # Rescale pixels

    # Make prediction
    predictions = model.predict(img_array)
    predicted_class_index = np.argmax(predictions[0])
    confidence = np.max(predictions[0]) * 100

    predicted_class_name = class_names[predicted_class_index]

    # Show results
    st.success(f"**Prediction:** {predicted_class_name}")
    st.info(f"**Confidence:** {confidence:.2f}%")

Overwriting app.py


In [32]:
# Install pyngrok
!pip install -q pyngrok

In [ ]:
# Install localtunnel properly
!npm install -g localtunnel

# Run streamlit in background and run localtunnel on port 8501
import subprocess
import threading

def run_streamlit():
    subprocess.run(["streamlit", "run", "app.py"])

# Start streamlit in a separate thread
threading.Thread(target=run_streamlit).start()

# Print external IP and start localtunnel
import urllib.request
external_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"Your External IP for localtunnel is: {external_ip}")

!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
changed 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦Your External IP for localtunnel is: 136.69.186.142
⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://lazy-planes-jam.loca.lt


In [ ]:
%%writefile README.md
# Tomato Plant Disease Detection

A complete Deep Learning predictive web application built to classify tomato leaf diseases into 10 distinct categories (9 common diseases + 1 healthy class) using Transfer Learning (MobileNetV2) and Streamlit.

## Dataset
- **PlantVillage Dataset** (Color images of tomato leaves).
- 10 Classes covering healthy leaves and various conditions (e.g., Early Blight, Late Blight, Bacterial Spot, Yellow Leaf Curl Virus, etc.).

## Technologies Used
- **Google Colab** & **Python**
- **TensorFlow / Keras** (MobileNetV2 for Feature Extraction & Transfer Learning)
- **Streamlit** (Interactive Web App Interface)
- **Pandas & NumPy** (Data Preprocessing)

## Model Performance
- Achieved a validation accuracy (`val_accuracy`) of over **91.2%** after training.